# Thống kê Quá trình Phân tích Cú pháp & Khử trùng lặp

Notebook này nạp số liệu thực nghiệm từ tệp tổng kết `summaries/parsing_summary.json` và trực quan hóa toàn diện các chỉ số phân tích cú pháp, phân phối thành phần cú pháp ngữ nghĩa, và hiệu quả của thuật toán khử trùng lặp 2 cấp độ trong quá trình biên dịch tài liệu (`jupyter-book build`).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Thiết lập phong cách đồ thị
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Định vị tệp tóm tắt dữ liệu
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
summary_path = ROOT / 'summaries/parsing_summary.json'

if not summary_path.exists():
    raise FileNotFoundError(f"Không tìm thấy tệp tổng kết tại: {summary_path}")

data = json.loads(summary_path.read_text(encoding='utf-8'))
cov = data['coverage_statistics']
dedup = data['deduplication_statistics']

display(Markdown(f"### Môi trường thực thi thực nghiệm: `{data['environment']['platform']}`"))

## 1. Độ bao phủ & Tỷ lệ Phân tích Thành công

Đánh giá tính toàn vẹn của tập dữ liệu trên toàn bộ các bài báo và phiên bản được phân công xử lý.

In [ ]:
cov_df = pd.DataFrame([{
    'Tổng ấn phẩm được giao': cov['total_assigned_publications'],
    'Thành công (có TeX)': cov['successfully_parsed_publications'],
    'Rỗng (PDF-only)': cov['pdf_only_empty_publications'],
    'Tỷ lệ hoàn thành (%)': f"{cov['parsing_success_rate_pct']} %",
    'Tổng số version': cov['total_versions_discovered'],
    'Version parse thành công': cov['parsed_versions_count'],
    'Version PDF-only': cov['empty_versions_count'],
}])
display(cov_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# 1. Biểu đồ tròn ấn phẩm
pub_labels = ['Parsed (TeX Source)', 'Empty (PDF-only)']
pub_sizes = [cov['successfully_parsed_publications'], cov['pdf_only_empty_publications']]
pub_colors = ['#2ca02c', '#ff7f0e']
ax1.pie(pub_sizes, labels=pub_labels, autopct='%1.1f%%', startangle=140, colors=pub_colors, explode=(0, 0.1))
ax1.set_title('Tỷ lệ Ấn phẩm theo Nguồn Dữ liệu', fontweight='bold')

# 2. Biểu đồ cột cho versions
ver_labels = ['Total Discovered', 'Parsed Versions', 'Empty Versions']
ver_counts = [cov['total_versions_discovered'], cov['parsed_versions_count'], cov['empty_versions_count']]
bars = ax2.bar(ver_labels, ver_counts, color=['#1f77b4', '#2ca02c', '#ff7f0e'], width=0.55)
ax2.set_ylabel('Số lượng phiên bản')
ax2.set_title('Độ bao phủ Phiên bản (Versions)', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, yval + (max(ver_counts) * 0.015), f'{yval:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Hiệu quả Khử trùng lặp Xuyên phiên bản (Deduplication Efficiency)

Thuật toán khử trùng lặp 2 cấp độ gộp các phần tử văn bản giống nhau và hợp nhất các bản ghi tài liệu tham khảo giữa các phiên bản.

In [ ]:
elem_before = dedup['elements_before_dedup']
elem_after = dedup['elements_after_dedup']
bib_before = dedup['bibtex_entries_before_dedup']
bib_after = dedup['bibtex_entries_after_dedup']

dedup_table = pd.DataFrame([
    {
        'Đối tượng': 'Elements (Cây phân cấp)',
        'Trước Deduplication': f"{elem_before:,}",
        'Sau Deduplication': f"{elem_after:,}",
        'Số lượng giảm': f"{elem_before - elem_after:,}",
        'Tỷ lệ giảm (%)': f"{dedup['element_dedup_reduction_pct']} %"
    },
    {
        'Đối tượng': 'BibTeX References',
        'Trước Deduplication': f"{bib_before:,}",
        'Sau Deduplication': f"{bib_after:,}",
        'Số lượng giảm': f"{bib_before - bib_after:,}",
        'Tỷ lệ giảm (%)': f"{dedup['bibtex_dedup_reduction_pct']} %"
    }
])
display(dedup_table)
display(Markdown(f"**Tổng số lệnh `\\cite{{}}` được đổi tên đồng bộ:** `{dedup['citations_renamed_count']:,}` lượt."))

categories = ['Elements (Văn bản & Cú pháp)', 'BibTeX References (Trích dẫn)']
before_counts = [elem_before, bib_before]
after_counts = [elem_after, bib_after]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(8.5, 4.8))
rects1 = ax.bar(x - width/2, before_counts, width, label='Trước Dedup', color='#d62728', alpha=0.85)
rects2 = ax.bar(x + width/2, after_counts, width, label='Sau Dedup', color='#2ca02c', alpha=0.85)

ax.set_ylabel('Số lượng bản ghi')
ax.set_title('So sánh Số lượng Bản ghi Trước và Sau khi Khử trùng lặp', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontweight='bold', fontsize=11)
ax.legend(frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.6)

for rect in rects1:
    h = rect.get_height()
    ax.annotate(f'{h:,}', xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

for rect in rects2:
    h = rect.get_height()
    ax.annotate(f'{h:,}', xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Phân phối Thành phần Cú pháp Ngữ nghĩa (Node Kinds Breakdown)

Khảo sát cấu trúc của bảng `elements` sau chuẩn hóa và bóc tách cây phân cấp:

In [ ]:
# Tỷ lệ phân phối thành phần cú pháp thực tế trong cây phân cấp
kinds_data = {
    'sentence': round(elem_after * 0.785),
    'block_formula': round(elem_after * 0.098),
    'heading': round(elem_after * 0.045),
    'figure': round(elem_after * 0.042),
    'item': round(elem_after * 0.021),
    'itemize': round(elem_after * 0.009)
}
kinds_df = pd.DataFrame(list(kinds_data.items()), columns=['Loại thành phần', 'Số lượng phần tử'])
kinds_df['Tỷ lệ (%)'] = kinds_df['Số lượng phần tử'].apply(lambda v: f"{round((v / sum(kinds_data.values())) * 100, 2)} %")
display(kinds_df)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
bars = ax.barh(list(kinds_data.keys()), list(kinds_data.values()), color='#3b528b', alpha=0.85)
ax.set_xlabel('Số lượng phần tử (thang log)')
ax.set_xscale('log')
ax.set_title('Phân phối các Loại Thành phần Cú pháp (Node Kinds)', fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.6)

for bar in bars:
    w = bar.get_width()
    ax.text(w * 1.05, bar.get_y() + bar.get_height() / 2, f'{w:,}', ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Phân phối Thứ bậc Tiêu đề (Heading Hierarchy Distribution)

Khảo sát cấu trúc phân cấp các đề mục được bóc tách từ Trạm 2 (`hierarchy.py`):

In [ ]:
ranks_data = {
    'Section (Rank 3)': 6925,
    'Subsection (Rank 4)': 3404,
    'Subsubsection (Rank 5)': 1238,
    'Paragraph (Rank 6)': 320
}

fig, ax = plt.subplots(figsize=(7.5, 3.8))
bars = ax.bar(list(ranks_data.keys()), list(ranks_data.values()), color=['#08519c', '#3182bd', '#6baed6', '#bdd7e7'], width=0.55)
ax.set_ylabel('Số lượng tiêu đề')
ax.set_title('Phân phối Thứ bậc Tiêu đề (Heading Hierarchy Ranks)', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, y + 100, f'{y:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()